In [1]:
from datasets import load_dataset
from utils import readJson, writeJson
import os
from sentence_transformers import SentenceTransformer, losses, InputExample
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

C:\Users\nosen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def componiTextPlayerMatch(player_match: dict) -> str:
    text = ''
    i = 0
    for id, action in player_match['text'].items():
        if i > 0:
            if int(id) == i+1:
                text = text +', ' + action
            else:
                text = text +'. ' + action
        else:
            text = action
        i = int(id)

    return text

In [4]:
src_dir = 'Dataset\\Events2Text\\PlayerDocs'
tgt_dir = 'Dataset\\Events2Text'
entire_dataset = []
j=0
for p in os.listdir(src_dir):
    src_dir_p = os.path.join(src_dir, p)
    for s in os.listdir(src_dir_p):
        src_dir_p_s = os.path.join(src_dir_p, s)
        for match in os.listdir(src_dir_p_s):
            player_match = readJson(os.path.join(src_dir_p_s, match))
            teamId = list(player_match['teamId'].values())[0]
            team = list(player_match['team'].values())[0]
            playerId = list(player_match['playerId'].values())[0]
            playerName = list(player_match['playerName'].values())[0]
            record = dict(season=s, playerId=playerId, playerName=playerName, teamId=teamId, teamName=team)
            record['text'] = componiTextPlayerMatch(player_match)
            record['match'] = match
            entire_dataset.append(record)

writeJson(entire_dataset, os.path.join(tgt_dir, 'player2vec_dataset.json'))

In [71]:
dataset = load_dataset('json', data_files='Dataset/Events2Text/player2vec_dataset.json', streaming=True)

In [27]:
list_dataset = readJson( os.path.join(tgt_dir, 'player2vec_dataset.json'))
for row in list_dataset:
    row["match"] = row["match"][:-5]
    row.pop(match)

writeJson(list_dataset, os.path.join(tgt_dir, 'player2vec_dataset.json'))

In [76]:
#train_dataset = dataset['train'].shuffle(buffer_size=10_000).take(50_000) 
def generate_examples(dataset):
    for row in dataset['train']:
        yield row['text']  #

In [7]:
from torch.utils.data import DataLoader, IterableDataset
class StreamingDataset(IterableDataset):
    def __init__(self, dataset, batch_size = 30):
        self.dataset = dataset
        
        self.batch_size = batch_size
        self.buffer = []
    
    def __iter__(self):
        for row in self.dataset['train']:
            # Store sentences in buffer until the batch size is met
            self.buffer.append(row['text'])
            if len(self.buffer) >= self.batch_size:
                yield self.buffer  # Yield a batch of sentences
                self.buffer = []  # Reset buffer

        # Yield the last remaining batch (if any)
        if len(self.buffer) > 0:
            yield self.buffer

In [9]:
dataset = load_dataset('json', data_files='Dataset/Events2Text/player2vec_dataset.json')
model = SentenceTransformer("distiluse-base-multilingual-cased-v1", device=device)
train_examples = dataset['train'][:]['text']
batch = 30
#train_examples = StreamingDataset(dataset, batch_size=batch)
#train_examples = generate_examples(dataset)
train_dataloader = losses.ContrastiveTensionDataLoader(train_examples, batch_size=batch, pos_neg_ratio=3)
train_loss = losses.ContrastiveTensionLoss(model=model)
model.fit(
    [(train_dataloader, train_loss)],
    epochs=10,
    output_path="Models/playermatch2vec",
    show_progress_bar=True
)

  1%|          | 500/80080 [1:22:15<226:25:03, 10.24s/it]

{'loss': 20.7596, 'grad_norm': 31.088499069213867, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.06}


  1%|          | 1000/80080 [2:44:33<220:37:09, 10.04s/it]

{'loss': 16.8463, 'grad_norm': 38.625003814697266, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.12}


  2%|▏         | 1500/80080 [4:06:59<224:33:27, 10.29s/it]

...

{'loss': 8.1523, 'grad_norm': 29.218685150146484, 'learning_rate': 1.5000000000000002e-05, 'epoch': 0.94}


 10%|▉         | 8000/80080 [21:54:34<191:30:52,  9.57s/it]

{'loss': 7.7183, 'grad_norm': 26.262022018432617, 'learning_rate': 1.6000000000000003e-05, 'epoch': 1.0}


 10%|█         | 8022/80080 [21:58:43<197:01:13,  9.84s/it]

KeyboardInterrupt: 

In [ ]:
dataset

IterableDataset({
    features: ['season', 'playerId', 'playerName', 'teamId', 'teamName', 'text', 'match'],
    num_shards: 1
})

In [54]:
model.fit(
    [(train_dataloader, train_loss)],
    epochs=10,
)

MemoryError: 

In [57]:
chunk_size = 1000
chunks = [train_examples[i:i+chunk_size] for i in range(0, len(train_examples), chunk_size)]

'ino rosso)'